# Avance 2

### Conectarse a la API de Yelp para obtener información de negocios locales según la ciudad seleccionada.

In [42]:
#Importamos las librerías que se necesitan en este avance
import pandas as pd
import numpy as np
import requests
import os
#os: permite interactuar con variables de entorno del sistema, donde podemos guardar nuestra API Key.
#requests: permite enviar solicitudes HTTP a Yelp.

In [43]:
print(os.getenv("YELP_API_KEY") is not None)

True


In [44]:
# La clave se lee del secreto de Codespaces (YELP_API_KEY)
api_key = os.getenv("YELP_API_KEY")

if not api_key:
    raise ValueError("No se encontró YELP_API_KEY.")

#Endpoint de Yelp que vamos a utilizar para enviar la solicitud para buscar negocios en Yelp.
url = "https://api.yelp.com/v3/businesses/search"

# Autenticación: enviamos nuestra API Key a Yelp.
headers = {"Authorization": f"Bearer {api_key}"}

# Qué datos queremos obtener de Yelp
params = {
    "location": "Chicago",
    "categories": "restaurants",
    "limit": 50,    # máximo de resultados por solicitud
    "offset": 0     # desde qué resultado empezar
}

# Enviamos la solicitud y se hace la conexión
resp = requests.get(url, headers=headers, params=params, timeout=30)
print(resp.status_code)   # 200 = solicitud exitosa

if resp.status_code == 200:
    datos_api = resp.json()
    print("Total de negocios:", datos_api["total"])
    print(datos_api["businesses"][0])   # primer negocio, para ver los campos
else:
    print(resp.text)

200
Total de negocios: 7900
{'id': 'boE4Ahsssqic7o5wQLI04w', 'alias': 'the-purple-pig-chicago', 'name': 'The Purple Pig', 'image_url': 'https://s3-media0.fl.yelpcdn.com/bphoto/8993TKAeJ2rhDST6tuqh4A/o.jpg', 'is_closed': False, 'url': 'https://www.yelp.com/biz/the-purple-pig-chicago?adjust_creative=Sq3L8ThsRiXpDGDI5u46sw&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=Sq3L8ThsRiXpDGDI5u46sw', 'review_count': 9073, 'categories': [{'alias': 'tapasmallplates', 'title': 'Tapas/Small Plates'}, {'alias': 'mediterranean', 'title': 'Mediterranean'}, {'alias': 'newamerican', 'title': 'New American'}], 'rating': 4.3, 'coordinates': {'latitude': 41.890694, 'longitude': -87.624782}, 'transactions': ['delivery', 'pickup'], 'price': '$$$', 'location': {'address1': '444 N Michigan Ave', 'address2': '', 'address3': '', 'city': 'Chicago', 'zip_code': '60611', 'country': 'US', 'state': 'IL', 'display_address': ['444 N Michigan Ave', 'Chicago, IL 60611']}, 'phone': '+13124641744', 'd

In [45]:
#Veamos los keys del diccionario recibido
datos_api.keys()

dict_keys(['businesses', 'total', 'region'])

In [46]:
#El primer elemento del diccionario indica el total de restaurants existentes en la API
print(f"Yelp informa {datos_api['total']} resultados para la búsqueda de restaurantes en Chicago.")

Yelp informa 7900 resultados para la búsqueda de restaurantes en Chicago.


### Extraer datos como nombre del negocio, categoría, calificación, número de reseñas, calificación, etc. 

In [47]:
#Aqui insoeccionamos el restaurante 
restaurante = datos_api['businesses'][0]
#Esto mostrará los campos disponibles para el primer restaurante.
restaurante.keys()

dict_keys(['id', 'alias', 'name', 'image_url', 'is_closed', 'url', 'review_count', 'categories', 'rating', 'coordinates', 'transactions', 'price', 'location', 'phone', 'display_phone', 'distance', 'business_hours', 'attributes'])

In [48]:
# Creamos una lista para guardar los negocios obtenidos en cada solicitud
negocios = []

# Creamos un bucle para extraer los datos recorriendo cada página para tratar de obtener un total de 200 datos 
for offset in range(0, 200, 50):

#Enviamos la solicitud y la conviertimos la respuesta JSON en un diccionario de Python.
    params = {
        "location": "Chicago",
        "categories": "restaurants",
        "limit": 50,
        "offset": offset
    }

    # Nueva solicitud a Yelp
    resp = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=10
    )

    # Convertimos la respuesta a JSON
    datos_api = resp.json()

    print(resp.status_code)
    print(datos_api)

    #Agregamos a negocios cada elemento de la lista recibida. Asi guardamos los restaurantes de las distintas páginas en una sola lista.
    negocios.extend(datos_api["businesses"])

200
{'businesses': [{'id': 'boE4Ahsssqic7o5wQLI04w', 'alias': 'the-purple-pig-chicago', 'name': 'The Purple Pig', 'image_url': 'https://s3-media0.fl.yelpcdn.com/bphoto/8993TKAeJ2rhDST6tuqh4A/o.jpg', 'is_closed': False, 'url': 'https://www.yelp.com/biz/the-purple-pig-chicago?adjust_creative=Sq3L8ThsRiXpDGDI5u46sw&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=Sq3L8ThsRiXpDGDI5u46sw', 'review_count': 9073, 'categories': [{'alias': 'tapasmallplates', 'title': 'Tapas/Small Plates'}, {'alias': 'mediterranean', 'title': 'Mediterranean'}, {'alias': 'newamerican', 'title': 'New American'}], 'rating': 4.3, 'coordinates': {'latitude': 41.890694, 'longitude': -87.624782}, 'transactions': ['delivery', 'pickup'], 'price': '$$$', 'location': {'address1': '444 N Michigan Ave', 'address2': '', 'address3': '', 'city': 'Chicago', 'zip_code': '60611', 'country': 'US', 'state': 'IL', 'display_address': ['444 N Michigan Ave', 'Chicago, IL 60611']}, 'phone': '+13124641744', 'display_p

In [49]:
#revisamos los datos 
negocios[0]

{'id': 'boE4Ahsssqic7o5wQLI04w',
 'alias': 'the-purple-pig-chicago',
 'name': 'The Purple Pig',
 'image_url': 'https://s3-media0.fl.yelpcdn.com/bphoto/8993TKAeJ2rhDST6tuqh4A/o.jpg',
 'is_closed': False,
 'url': 'https://www.yelp.com/biz/the-purple-pig-chicago?adjust_creative=Sq3L8ThsRiXpDGDI5u46sw&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=Sq3L8ThsRiXpDGDI5u46sw',
 'review_count': 9073,
 'categories': [{'alias': 'tapasmallplates', 'title': 'Tapas/Small Plates'},
  {'alias': 'mediterranean', 'title': 'Mediterranean'},
  {'alias': 'newamerican', 'title': 'New American'}],
 'rating': 4.3,
 'coordinates': {'latitude': 41.890694, 'longitude': -87.624782},
 'transactions': ['delivery', 'pickup'],
 'price': '$$$',
 'location': {'address1': '444 N Michigan Ave',
  'address2': '',
  'address3': '',
  'city': 'Chicago',
  'zip_code': '60611',
  'country': 'US',
  'state': 'IL',
  'display_address': ['444 N Michigan Ave', 'Chicago, IL 60611']},
 'phone': '+13124641744',

In [50]:
# 1 Definimos las columnas
columnas = ["nombre", "categoria", "categorias", "rating", "reseñas", "precio"]

# 2 Creamos una lista vacía
datos = []

# 3 Recorremos los 200 negocios
for negocio in negocios:
    # 4 Extraemos los datos
    nombre = negocio["name"]
    lista_categorias = [c["title"] for c in negocio["categories"]]
    categoria = lista_categorias[0]               # categoría principal
    categorias = ", ".join(lista_categorias)      # todas las categorías, separadas por coma
    rating = negocio["rating"]
    reseñas = negocio["review_count"]
    precio = negocio.get("price")

    # 5 Guardamos los datos
    datos.append([nombre, categoria, categorias, rating, reseñas, precio])

# 6 Creamos el DataFrame
df_yelp = pd.DataFrame(datos, columns=columnas)

# 7 Visualizamos las primeras filas
df_yelp.head()

,nombre,categoria,categorias,rating,reseñas,precio
0,The Purple Pig,Tapas/Small Plates,"Tapas/Small Plates, Mediterranean, New American",4.3,9073,$$$
1,Cindy's Rooftop,New American,"New American, Seafood, Breakfast & Brunch",4.1,2902,$$$
2,Wildberry Pancakes and Cafe,Pancakes,"Pancakes, Waffles, Cafes",4.4,10451,$$
3,Pequod's Pizza,Pizza,Pizza,3.9,9024,$$
4,Trivoli Tavern,American,"American, Desserts, Breakfast & Brunch",4.5,1065,$$


In [61]:
df_yelp.head()

,nombre,categoria,categorias,rating,reseñas,precio,ciudad
0,The Purple Pig,Tapas/Small Plates,"Tapas/Small Plates, Mediterranean, New American",4.3,9073,$$$,Chicago
1,Cindy's Rooftop,New American,"New American, Seafood, Breakfast & Brunch",4.1,2902,$$$,Chicago
2,Wildberry Pancakes and Cafe,Pancakes,"Pancakes, Waffles, Cafes",4.4,10451,$$,Chicago
3,Pequod's Pizza,Pizza,Pizza,3.9,9024,$$,Chicago
4,Trivoli Tavern,American,"American, Desserts, Breakfast & Brunch",4.5,1065,$$,Chicago


In [51]:
#Vemos como están los datos 
df_yelp.info()
#Podemos ver que la información nos muestra cuatro columnas en las cuales no existen nulos pero si existe 46 datos faltantes en precio

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   nombre      200 non-null    str    
 1   categoria   200 non-null    str    
 2   categorias  200 non-null    str    
 3   rating      200 non-null    float64
 4   reseñas     200 non-null    int64  
 5   precio      152 non-null    str    
dtypes: float64(1), int64(1), str(4)
memory usage: 9.5 KB


In [ ]:
#Aqui ve el porcentaje del promedio de valores faltantes
df_yelp["precio"].isnull().mean() * 100
#El 24 % de los restaurantes carece de información de precio.

np.float64(24.0)

In [ ]:
#Vamos a verificar como esta el dataframe
df_yelp.shape
#Tenemos 200 filas y 6 columnas

(200, 6)

In [54]:
df_yelp.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   nombre      200 non-null    str    
 1   categoria   200 non-null    str    
 2   categorias  200 non-null    str    
 3   rating      200 non-null    float64
 4   reseñas     200 non-null    int64  
 5   precio      152 non-null    str    
dtypes: float64(1), int64(1), str(4)
memory usage: 9.5 KB


In [55]:
df_yelp.head()

,nombre,categoria,categorias,rating,reseñas,precio
0,The Purple Pig,Tapas/Small Plates,"Tapas/Small Plates, Mediterranean, New American",4.3,9073,$$$
1,Cindy's Rooftop,New American,"New American, Seafood, Breakfast & Brunch",4.1,2902,$$$
2,Wildberry Pancakes and Cafe,Pancakes,"Pancakes, Waffles, Cafes",4.4,10451,$$
3,Pequod's Pizza,Pizza,Pizza,3.9,9024,$$
4,Trivoli Tavern,American,"American, Desserts, Breakfast & Brunch",4.5,1065,$$


In [56]:
#Vamos a comprobar si es que hay duplicados
print('Filas duplicadas:', df_yelp.duplicated().sum())
print('Nombres repetidos:', df_yelp['nombre'].duplicated().sum())
df_yelp[df_yelp['nombre'].duplicated(keep=False)].sort_values('nombre')

Filas duplicadas: 0
Nombres repetidos: 1


,nombre,categoria,categorias,rating,reseñas,precio
11,Qing Xiang Yuan Dumplings,Chinese,"Chinese, Barbeque",4.4,2220,$$
62,Qing Xiang Yuan Dumplings,Dim Sum,"Dim Sum, Barbeque, Coffee & Tea",4.6,35,NaN


In [57]:
#Aquí vemos que no hay valores nulos
df_yelp['precio'] = df_yelp['precio'].fillna('No especificado')
df_yelp['ciudad'] = 'Chicago'
df_yelp.isna().sum()

nombre        0
categoria     0
categorias    0
rating        0
reseñas       0
precio        0
ciudad        0
dtype: int64

In [ ]:
#Ahora vemos los porcentajes de cada precio
df_yelp['precio'].value_counts(normalize=True) * 100
#Tenemos que el 55% de los restaurantes es `$$`; solo 3.5 son `$` y 3 son `$$$$`. el 24% no tiene dato de precio.

precio
$$                 55.0
No especificado    24.0
$$$                14.5
$                   3.5
$$$$                3.0
Name: proportion, dtype: float64

In [ ]:
df_yelp['categoria'].value_counts(normalize=True).head(15)*100

categoria
Pizza                 4.5
Breakfast & Brunch    4.0
Mexican               4.0
Latin American        3.5
Thai                  3.5
Japanese              3.5
New American          3.0
Mediterranean         3.0
Asian Fusion          3.0
American              2.5
Italian               2.5
Sushi Bars            2.5
Cocktail Bars         2.5
Steakhouses           2.0
Barbeque              2.0
Name: proportion, dtype: float64

In [ ]:
#Aqui vemos las categorias unicas
df_yelp['categoria'].nunique()
#- Los 200 restaurantes tienen 86 categorías distintas, por lo que se agruparán en familias más amplias 
# para poder compararlas con las preferencias alimenticias de los clientes.

86

### Exploración de los datos de Yelp
- Se obtuvieron 200 restaurantes de Chicago, sin duplicados.
- La calificación es alta y homogénea (mediana 4,4; rango 3,6 a 5,0).
- Las reseñas tienen una distribución muy sesgada (mediana 385, máximo 10.451), por lo que se usa la mediana

### Integrar esta información al análisis que se está realizando para una ciudad  en específico. 